# 7.1 相关随机变量（Dependent Random Variables）

### ML任务的基础

* **Dependence**  
  * Classification / regression and similar problems need it  
    $p(y \mid x) = \frac{p(x, y)}{p(x)}$
  * A/B testing, cause / effect …  

* **Independence**  $p(x, y) = p(x) \cdot p(y)$
  * Can ignore variables $p(y \mid x) = \frac{p(x, y)}{p(x)} = \frac{p(x) \cdot p(y)}{p(x)} = p(y)$

## 独立性检验

### 分类器
- 正样本（真组合）：直接使用原始配对数据 $(x_i, y_i)$。这些样本代表了联合分布 $p(x, y)$。
- 负样本（假组合）：保持 $x_i$ 不变，但将 $y$ 的顺序彻底打乱（Shuffle），得到 $(x_i, y_{\pi(i)})$。这些样本代表了独立分布的乘积 $p(x)p(y)$。

- 给“真组合”贴上标签 $1$，给“假组合”贴上标签 $0$。训练一个分类器区分两类

- 观察分类器在测试集上的表现：
    - 准确率接近 50%, 两者独立, 分类器在瞎猜，说明真假组合长得完全一样，p(x,y)≈p(x)p(y)
    - 准确率**显著高于 50%**,分类器能看出某种“配对模式”，说明 x 和 y 之间存在某种联系。

- 对于相关数据：
    - A Crazy Idea：$\hat{y} = \text{argmax}_y f(x, y)$，对于一个新的输入 $x$，只要尝试所有的 $y$，看哪一个能让分类器给出最高分，就能完成预测（y空间可能太大）
    - Not so crazy：对比学习（Contrastive Learning）：在训练阶段利用这个性质，对于正确的配对 $(x, y)$，让模型输出高分（$f(x, y) > 0$）；而对所有错误的配对 $(x, y')$，强制模型输出低分（$f(x, y') < 0$）

**扩展：对比学习**

- 在没有人工标签的情况下，通过数据自身的“自我对比”来学习有用的特征表示。
- 对比学习的三大支柱
    - 数据增强 (Data Augmentation)： 为了创造出“正样本对”，对同一个样本进行两次不同的随机变换。
    - 编码器 (Encoder)： 目标模型，输入一个样本对 $x_i$ 和 $x_j$，输出两个高维向量 $h_i$ 和 $h_j$
    - 投影头 (Projection Head)：将特征 $h_i$ 和 $h_j$ 映射到另一个空间得到 $z_i$ 和 $z_j$，并在这个空间计算两个向量的相似度
    - 对比损失函数 (Contrastive Loss)：InfoNCE Loss：$$\ell_{i,j} = -\log \frac{\exp(\text{sim}(z_i, z_j) / \tau)}{\sum_{k=1}^{2N} \mathbb{1}_{[k \neq i]} \exp(\text{sim}(z_i, z_k) / \tau)}$$
        - 温度参数 ($\tau$)：用来控制模型对“困难样本”的敏感程度。
        - 分子 (Numerator)：计算 $z_i$ 和 $z_j$（正样本对）的相似度。我们希望这个值越大越好（拉近距离）
        - 分母 (Denominator)：计算 $z_i$ 与所有样本（包括正样本对和所有负样本）的相似度之和，只有推开 $z_i$ 与负样本的距离才能越小

### 最大均值差异 (Maximum Mean Discrepancy, MMD)
比较联合分布和边缘分布乘积的均值距离
$$\left\lVert \underbrace{\mathbf{E}_{(x,y)}\left[\phi(x) \cdot \phi(y)\right]}_{\text{Joint expectation}} - \underbrace{\mathbf{E}_x \mathbf{E}_y \left[\phi(x) \cdot \phi(y)\right]}_{\text{Independent expectation}} \right\rVert^2$$
$$f(x, y) = \frac{1}{m} \sum_{i=1}^m k(x_i, x) l(y_i, y) - \frac{1}{m^2} \sum_{i=1}^m k(x_i, x) \sum_{j=1}^m l(y_j, y)$$

### 希尔伯特-施密特独立性指标 (Hilbert-Schmidt Independence Criterion, HSIC)
- **Covariance operator** (like covariance matrix) should vanish  

- 基础统计学中，用协方差 (Covariance) 来衡量两个变量是否同步变化，但只能捕捉线性关系。
- HSIC 通过核函数将变量映射到高维空间，然后计算一个“协方差算子”的大小
$$HSIC(P_{xy}) = \left\lVert \operatorname{Cov}_{(x,y)}\!\left[\phi(x), \phi(y)\right] \right\rVert^2  \\=  \left\lVert \mathbb{E}_{(x,y)} \left[ \left( \phi(x) - \mathbb{E}_{x'}\!\left[\phi(x')\right] \right) \cdot \left( \phi(y) - \mathbb{E}_{y'}\!\left[\phi(y')\right] \right) \right] \right\rVert^2 \\
= \left\lVert \mathbb{E}_{(x,y)}\!\left[\phi(x) \cdot \phi(y)\right] - \mathbb{E}_x \mathbb{E}_y \!\left[\phi(x) \cdot \phi(y)\right] \right\rVert^2$$

和 MMD 同样的形式

$$\operatorname{tr}(HKHL) \quad \text{where} \quad H_{ij} = \delta_{ij} - m^{-1}, \; K_{ij} = k(x_i, x_j), \; \text{and} \; L_{ij} = l(y_i, y_j)$$

### Information Theory：Kullback-Leibler Divergence
$$\begin{align*}
D\left(p(x,y) \parallel p(x)p(y)\right) &= \int dp(x,y)\left[\log p(x,y) - \log\left[p(x)p(y)\right]\right] \\
&= H[y] + H[x] - H[(x,y)] \\
&= I(x,y)
\end{align*}$$

- Count number of extra bits required to encode X and Y relative to encoding them jointly.
- If the data is independent, no bits can be saved.